In [1]:
import os
import pandas as pd
from pathlib import Path

# ─── CONFIG ───────────────────────────────────────────────────────────────────
CSV_PATH    = r"C:\Users\Tanishq\Downloads\final_paired_dataset_advance.csv"
MRI_ROOT    = r"C:\Users\Tanishq\Downloads\MRI\ADNI"
PET_ROOT    = r"C:\Users\Tanishq\Downloads\PET2\ADNI"
# ──────────────────────────────────────────────────────────────────────────────

def find_image_folder(root, subject_id, image_id):
    """Search for I{image_id} folder under root/subject_id recursively."""
    subject_path = Path(root) / subject_id
    target = f"I{image_id}"
    if not subject_path.exists():
        return None, "subject_folder_missing"
    for folder in subject_path.rglob(target):
        if folder.is_dir():
            return folder, "found"
    return None, "image_folder_missing"

def count_dcm(folder):
    """Count .dcm files in a folder."""
    if folder is None:
        return 0
    return len(list(folder.glob("*.dcm")))

# Load CSV
df = pd.read_csv(CSV_PATH)
print(f"Total pairs in CSV: {len(df)}\n")

results = []
missing_mri = []
missing_pet = []
empty_mri   = []
empty_pet   = []

for idx, row in df.iterrows():
    subject_id   = str(row["Subject_ID"]).strip()
    mri_image_id = str(int(row["MRI_Image_ID"]))
    pet_image_id = str(int(row["PET_Image_ID"]))
    label        = row["Class_Label"]

    # Find MRI
    mri_folder, mri_status = find_image_folder(MRI_ROOT, subject_id, mri_image_id)
    mri_dcm_count = count_dcm(mri_folder)

    # Find PET
    pet_folder, pet_status = find_image_folder(PET_ROOT, subject_id, pet_image_id)
    pet_dcm_count = count_dcm(pet_folder)

    # Flag issues
    if mri_status != "found":
        missing_mri.append((subject_id, mri_image_id, mri_status))
    elif mri_dcm_count == 0:
        empty_mri.append((subject_id, mri_image_id, str(mri_folder)))

    if pet_status != "found":
        missing_pet.append((subject_id, pet_image_id, pet_status))
    elif pet_dcm_count == 0:
        empty_pet.append((subject_id, pet_image_id, str(pet_folder)))

    results.append({
        "Subject_ID"    : subject_id,
        "Class_Label"   : label,
        "MRI_Image_ID"  : mri_image_id,
        "MRI_Status"    : mri_status,
        "MRI_DCM_Count" : mri_dcm_count,
        "MRI_Path"      : str(mri_folder) if mri_folder else "N/A",
        "PET_Image_ID"  : pet_image_id,
        "PET_Status"    : pet_status,
        "PET_DCM_Count" : pet_dcm_count,
        "PET_Path"      : str(pet_folder) if pet_folder else "N/A",
    })

# ─── SUMMARY ──────────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results)

ok_pairs = results_df[
    (results_df["MRI_Status"] == "found") &
    (results_df["MRI_DCM_Count"] > 0) &
    (results_df["PET_Status"] == "found") &
    (results_df["PET_DCM_Count"] > 0)
]

print("=" * 60)
print("AUDIT SUMMARY")
print("=" * 60)
print(f"Total pairs          : {len(df)}")
print(f"✅ Fully OK pairs    : {len(ok_pairs)}")
print(f"❌ Missing MRI       : {len(missing_mri)}")
print(f"❌ Missing PET       : {len(missing_pet)}")
print(f"⚠️  Empty MRI folder : {len(empty_mri)}")
print(f"⚠️  Empty PET folder : {len(empty_pet)}")
print("=" * 60)

if missing_mri:
    print("\n--- Missing MRI ---")
    for s, i, reason in missing_mri:
        print(f"  Subject: {s}  |  Image_ID: I{i}  |  Reason: {reason}")

if missing_pet:
    print("\n--- Missing PET ---")
    for s, i, reason in missing_pet:
        print(f"  Subject: {s}  |  Image_ID: I{i}  |  Reason: {reason}")

if empty_mri:
    print("\n--- Empty MRI folders (0 DCM files) ---")
    for s, i, path in empty_mri:
        print(f"  Subject: {s}  |  Image_ID: I{i}  |  Path: {path}")

if empty_pet:
    print("\n--- Empty PET folders (0 DCM files) ---")
    for s, i, path in empty_pet:
        print(f"  Subject: {s}  |  Image_ID: I{i}  |  Path: {path}")

# Save full audit log
out_path = r"C:\Users\Tanishq\Downloads\audit_results.csv"
results_df.to_csv(out_path, index=False)
print(f"\n📄 Full audit saved to: {out_path}")

Total pairs in CSV: 762

AUDIT SUMMARY
Total pairs          : 762
✅ Fully OK pairs    : 467
❌ Missing MRI       : 0
❌ Missing PET       : 0
⚠️  Empty MRI folder : 0
⚠️  Empty PET folder : 295

--- Empty PET folders (0 DCM files) ---
  Subject: 006_S_4150  |  Image_ID: I393812  |  Path: C:\Users\Tanishq\Downloads\PET2\ADNI\006_S_4150\ADNI_FDG_6F_4i_16s\2013-10-09_07_35_11.0\I393812
  Subject: 006_S_4153  |  Image_ID: I391528  |  Path: C:\Users\Tanishq\Downloads\PET2\ADNI\006_S_4153\ADNI_FDG_6F_4i_16s\2013-09-19_07_38_09.0\I391528
  Subject: 006_S_4192  |  Image_ID: I393810  |  Path: C:\Users\Tanishq\Downloads\PET2\ADNI\006_S_4192\ADNI_FDG_6F_4i_16s\2013-10-09_05_59_08.0\I393810
  Subject: 006_S_4357  |  Image_ID: I414313  |  Path: C:\Users\Tanishq\Downloads\PET2\ADNI\006_S_4357\ADNI_FDG_6F_4i_16s\2014-02-10_07_31_07.0\I414313
  Subject: 006_S_4485  |  Image_ID: I417790  |  Path: C:\Users\Tanishq\Downloads\PET2\ADNI\006_S_4485\ADNI_FDG_6F_4i_16s\2014-03-21_07_39_09.0\I417790
  Subject: 0

In [3]:
import os
from pathlib import Path

PET_ROOT = r"C:\Users\Tanishq\Downloads\PET2\ADNI"

# Pick one of the empty PET folder paths from audit_results.csv and check it
import pandas as pd
df = pd.read_csv(r"C:\Users\Tanishq\Downloads\audit_results.csv")
empty = df[df["PET_DCM_Count"] == 0]

# Check first 5 empty ones
for _, row in empty.head(5).iterrows():
    path = Path(row["PET_Path"])
    print(f"\nPath: {path}")
    print(f"Exists: {path.exists()}")
    if path.exists():
        all_files = list(path.rglob("*"))
        print(f"All files inside: {all_files[:10]}")


Path: C:\Users\Tanishq\Downloads\PET2\ADNI\006_S_4150\ADNI_FDG_6F_4i_16s\2013-10-09_07_35_11.0\I393812
Exists: True
All files inside: [WindowsPath('C:/Users/Tanishq/Downloads/PET2/ADNI/006_S_4150/ADNI_FDG_6F_4i_16s/2013-10-09_07_35_11.0/I393812/ADNI_006_S_4150_PET_ADNI_FDG_6F_4i_16s__br_raw_20131009135644638_S203398_I393812.v')]

Path: C:\Users\Tanishq\Downloads\PET2\ADNI\006_S_4153\ADNI_FDG_6F_4i_16s\2013-09-19_07_38_09.0\I391528
Exists: True
All files inside: [WindowsPath('C:/Users/Tanishq/Downloads/PET2/ADNI/006_S_4153/ADNI_FDG_6F_4i_16s/2013-09-19_07_38_09.0/I391528/ADNI_006_S_4153_PET_ADNI_FDG_6F_4i_16s__br_raw_20130923153936706_S201813_I391528.v')]

Path: C:\Users\Tanishq\Downloads\PET2\ADNI\006_S_4192\ADNI_FDG_6F_4i_16s\2013-10-09_05_59_08.0\I393810
Exists: True
All files inside: [WindowsPath('C:/Users/Tanishq/Downloads/PET2/ADNI/006_S_4192/ADNI_FDG_6F_4i_16s/2013-10-09_05_59_08.0/I393810/ADNI_006_S_4192_PET_ADNI_FDG_6F_4i_16s__br_raw_20131009135509696_S203396_I393810.v')]

Pat

In [5]:
import pandas as pd

# Load audit results
df = pd.read_csv(r"C:\Users\Tanishq\Downloads\audit_results.csv")

# Keep only fully OK pairs (both MRI and PET have DCM files)
ok = df[(df["MRI_DCM_Count"] > 0) & (df["PET_DCM_Count"] > 0)].copy()
ok = ok.reset_index(drop=True)

# Keep only useful columns
ok = ok[[
    "Subject_ID",
    "Class_Label",
    "MRI_Image_ID",
    "MRI_Path",
    "MRI_DCM_Count",
    "PET_Image_ID",
    "PET_Path",
    "PET_DCM_Count",
]]

# Add numeric label
label_map = {"CN": 0, "MCI": 1, "AD": 2}
ok["Numeric_Label"] = ok["Class_Label"].map(label_map)

# Save
out_path = r"C:\Users\Tanishq\Downloads\final_dataset_467.csv"
ok.to_csv(out_path, index=False)

# Summary
print("=" * 50)
print(f"Total OK pairs saved : {len(ok)}")
print("\nClass distribution:")
print(ok["Class_Label"].value_counts())
print(f"\n✅ Saved to: {out_path}")

Total OK pairs saved : 467

Class distribution:
Class_Label
MCI    206
CN     198
AD      63
Name: count, dtype: int64

✅ Saved to: C:\Users\Tanishq\Downloads\final_dataset_467.csv
